# Differentiable Sudoku Solver
## Gradient-Based Constraint Satisfaction on a 9×9×9 Soft Tensor

**Ports the hyperpoly-terrain 6-channel field computation architecture to a 9×9×9 probability lattice.**

| hyperpoly-terrain component | This Sudoku solver |
|-----------------------------|-------------------|
| 6-channel tensor (density, cohesion, perm, water, sediment, oxidation) | 9×9×9 probability tensor (81 cells × 9 digits) |
| QEF solver: minimize ‖Ax − b‖² with box constraints | Minimize constraint energy + clue fidelity |
| Two-pass dispatch (cull + solve) | Gradients propagate only through violated constraints |
| Material calibration (PBR → tensor init) | Given clues → softmax logits initialization |
| Vinculum ratio (top/bottom) | Solved / Total puzzles, confidence per cell |

**Lead R&D:** DaShawn (African American Developer & Mathematician)  
**Entity:** Guinea Pig Trench LLC  
**Copyright (c) 2026 Guinea Pig Trench LLC**

---

In [ ]:
# --- SETUP ---
import time, math, sys, json, warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

# GPU info
try:
    import subprocess
    r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.free','--format=csv,noheader'],
                       capture_output=True, text=True, timeout=5)
    GPU_INFO = r.stdout.strip() if r.returncode==0 else 'CPU'
except: GPU_INFO = 'CPU'

if torch.cuda.is_available():
    try:
        _ = torch.zeros(1, device='cuda')
        DEVICE = torch.device('cuda')
    except Exception as e:
        print(f"CUDA unavailable (incompatible GPU/driver): {e}")
        DEVICE = torch.device('cpu')
else:
    DEVICE = torch.device('cpu')
print(f"GPU: {GPU_INFO}")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"Start: {datetime.now().isoformat()}")
# --- SUBSTRATE DELTA SIEVE SETUP ---
try:
    sieve_dir = Path("C:/Users/dasha/Projects/SubstrateDeltaSieve")
    if str(sieve_dir) not in sys.path:
        sys.path.append(str(sieve_dir))
    from SUBSTRATE_DELTA_SIEVE import SubstrateDeltaSieve
    sieve = SubstrateDeltaSieve()
    print("SubstrateDeltaSieve loaded successfully!")
except Exception as e:
    print("Warning: Failed to load SubstrateDeltaSieve:", e)
    sieve = None


In [ ]:
# --- CONFIG ---
class Config:
    max_steps = 5000
    lr = 1.0
    anneal_start = 50
    anneal_end = 1000
    temp_start = 2.0
    temp_end = 0.001
    clue_weight = 5.0
    constraint_weight = 10.0
    sparsity_weight = 3.0
    tolerance = 1e-6

cfg = Config()
print(f"Config: max_steps={cfg.max_steps}, lr={cfg.lr}, temp={cfg.temp_start} -> {cfg.temp_end}")
print(f"         clue_weight={cfg.clue_weight}, constraint_weight={cfg.constraint_weight}, sparsity_weight={cfg.sparsity_weight}")

In [ ]:
# --- SUDOKU CONSTRAINTS (loss functions) ---

def uniqueness_loss(board):
    """
    Each digit must appear exactly once per row, column, and box.
    MSE on sums: sum(board over columns) should be 1.0 for each row-digit.
    """
    loss = 0.0
    row_sums = board.sum(dim=1)  # [9, 9]
    loss += ((row_sums - 1.0) ** 2).mean()
    col_sums = board.sum(dim=0)
    loss += ((col_sums - 1.0) ** 2).mean()
    for bi in range(3):
        for bj in range(3):
            box = board[bi*3:bi*3+3, bj*3:bj*3+3, :]
            box_sum = box.reshape(9, 9).sum(dim=0)
            loss += ((box_sum - 1.0) ** 2).mean()
    return loss / 3.0

def sparsity_loss(board):
    """Each cell should confidently pick one digit (one-hot)."""
    max_prob = board.max(dim=-1).values  # [9, 9]
    return (1.0 - max_prob).mean()  # 0 when all cells are one-hot

def clue_loss(board, clue_tensor):
    """Given clues should have high probability on the correct digit."""
    mask = (clue_tensor > 0).float()
    if mask.sum() == 0:
        return torch.tensor(0.0, device=board.device)
    clue_logits = torch.log(board.clamp(min=1e-8))
    loss = -(clue_logits * clue_tensor).sum(dim=-1)
    loss = (loss * mask).sum() / mask.sum()
    return loss

def total_energy(board, clue_tensor, cfg):
    return (cfg.clue_weight * clue_loss(board, clue_tensor)
            + cfg.constraint_weight * uniqueness_loss(board)
            + cfg.sparsity_weight * sparsity_loss(board))

In [ ]:
# --- DIFFERENTIABLE SUDOKU SOLVER (port of hyperpoly two-pass dispatch) ---

def solve_sudoku(clues, cfg, verbose=True):
    """Two-stage solver:
    1. Gradient descent on 9x9x9 soft tensor (port of hyperpoly field computation)
    2. Backtracking with gradient-derived heuristic ordering
    """
    clue_tensor = torch.zeros(9, 9, 9, device=DEVICE)
    for i in range(9):
        for j in range(9):
            d = clues[i][j]
            if d > 0:
                clue_tensor[i, j, d - 1] = 1.0

    # Initialize: uniform + noise, with clue boost
    logits = torch.randn(9, 9, 9, device=DEVICE) * 0.5
    logits = logits + clue_tensor * 8.0
    logits.requires_grad_(True)

    optimizer = torch.optim.Adam([logits], lr=cfg.lr)

    energy_history = []

    for step in range(cfg.max_steps):
        # Sieve dynamic pruning check (Sieve Pruning Test)
        if 'sieve' in globals() and sieve is not None:
            if not sieve.process_delta('symbolic', 2, step, "apparent", "hidden"):
                if verbose:
                    print(f"  [SIEVE BREACH] Modular breach detected at step {step}! Aborting execution branch.")
                break

        if step < cfg.anneal_start:
            temp = cfg.temp_start
        elif step > cfg.anneal_end:
            temp = cfg.temp_end
        else:
            frac = (step - cfg.anneal_start) / (cfg.anneal_end - cfg.anneal_start)
            temp = cfg.temp_start + (cfg.temp_end - cfg.temp_start) * frac

        board = F.softmax(logits / temp, dim=-1)
        energy = total_energy(board, clue_tensor, cfg)
        energy_history.append(energy.item())

        optimizer.zero_grad()
        energy.backward()
        optimizer.step()

        if len(energy_history) > 200 and abs(energy.item() - energy_history[-201]) < cfg.tolerance:
            if verbose:
                print(f"  Converged at step {step}, energy={energy.item():.4f}, temp={temp:.6f}")
            break

    # Read gradient solution
    with torch.no_grad():
        board_final = F.softmax(logits / cfg.temp_end, dim=-1)
        probs = board_final.cpu().numpy()  # [9, 9, 9]
        assignment = probs.argmax(axis=-1) + 1
        confidence = probs.max(axis=-1)

    # Stage 2: Backtracking solver with gradient heuristic ordering
    # Use original clues + fill remaining cells via search
    # Gradient solution provides candidate ordering (try most likely digits first)
    b = np.array([row[:] for row in clues])  # copy

    def valid(r, c, v):
        for i in range(9):
            if b[i][c] == v or b[r][i] == v:
                return False
        br, bc = 3 * (r // 3), 3 * (c // 3)
        for i in range(3):
            for j in range(3):
                if b[br+i][bc+j] == v:
                    return False
        return True

    def candidates():
        empty = []
        for i in range(9):
            for j in range(9):
                if b[i][j] == 0:
                    # Order by gradient probability descending
                    order = np.argsort(-probs[i, j]) + 1
                    possible = [v for v in order if valid(i, j, v)]
                    empty.append((len(possible), i, j, possible))
        empty.sort()
        return empty

    def search():
        emp = candidates()
        if not emp:
            return True
        _, r, c, possible = emp[0]
        if not possible:
            return False
        for v in possible:
            b[r][c] = v
            if search():
                return True
            b[r][c] = 0
        return False

    if search():
        assignment = np.array([row[:] for row in b])
        if verbose:
            print(f"  Stage 2: backtracking found valid solution")
    else:
        if verbose:
            print(f"  Stage 2: no valid completion found")

    return assignment, confidence, energy_history

In [ ]:
# --- VALIDATION ---

def validate_sudoku(board):
    def valid_group(g):
        return sorted(g) == list(range(1, 10))
    for i in range(9):
        if not valid_group(board[i].tolist()):
            return False
        if not valid_group(board[:, i].tolist()):
            return False
    for bi in range(0, 9, 3):
        for bj in range(0, 9, 3):
            box = board[bi:bi+3, bj:bj+3].flatten().tolist()
            if not valid_group(box):
                return False
    return True

def print_board(board, clues=None):
    sep = "+-------+-------+-------+"
    print(sep)
    for i in range(9):
        line = "| "
        for j in range(9):
            d = board[i][j]
            if clues is not None and clues[i][j] > 0:
                line += f"\033[92m{int(d)}\033[0m " if d > 0 else ". "
            else:
                line += f"{int(d)} " if d > 0 else ". "
            if (j + 1) % 3 == 0:
                line += "| "
        print(line)
        if (i + 1) % 3 == 0:
            print(sep)

print("Validation functions loaded.")

In [ ]:
# --- TEST PUZZLES (Wikipedia + hard examples) ---

PUZZLES = {
    "Easy (Wikipedia)": [
        [5, 3, 0, 0, 7, 0, 0, 0, 0],
        [6, 0, 0, 1, 9, 5, 0, 0, 0],
        [0, 9, 8, 0, 0, 0, 0, 6, 0],
        [8, 0, 0, 0, 6, 0, 0, 0, 3],
        [4, 0, 0, 8, 0, 3, 0, 0, 1],
        [7, 0, 0, 0, 2, 0, 0, 0, 6],
        [0, 6, 0, 0, 0, 0, 2, 8, 0],
        [0, 0, 0, 4, 1, 9, 0, 0, 5],
        [0, 0, 0, 0, 8, 0, 0, 7, 9],
    ],
    "Medium (Arto Inkala)": [
        [8, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 3, 6, 0, 0, 0, 0, 0],
        [0, 7, 0, 0, 9, 0, 2, 0, 0],
        [0, 5, 0, 0, 0, 7, 0, 0, 0],
        [0, 0, 0, 0, 4, 5, 7, 0, 0],
        [0, 0, 0, 1, 0, 0, 0, 3, 0],
        [0, 0, 1, 0, 0, 0, 0, 6, 8],
        [0, 0, 8, 5, 0, 0, 0, 1, 0],
        [0, 9, 0, 0, 0, 0, 4, 0, 0],
    ],
    "Hard (minimal ~17)": [
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 3, 0, 8, 5],
        [0, 0, 1, 0, 2, 0, 0, 0, 0],
        [0, 0, 0, 5, 0, 7, 0, 0, 0],
        [0, 0, 4, 0, 0, 0, 1, 0, 0],
        [0, 9, 0, 0, 0, 0, 0, 0, 0],
        [5, 0, 0, 0, 0, 0, 0, 7, 3],
        [0, 0, 2, 0, 1, 0, 0, 0, 0],
        [0, 0, 0, 0, 4, 0, 0, 0, 9],
    ],
    "Empty (stress test)": [
        [0]*9 for _ in range(9)
    ],
}

print(f"Loaded {len(PUZZLES)} test puzzles:")
for name in PUZZLES:
    clues_count = sum(1 for row in PUZZLES[name] for c in row if c > 0)
    print(f"  {name}: {clues_count} clues")

In [ ]:
# --- RUN THE SOLVER ---

results = []

for name, clues in PUZZLES.items():
    print(f"\n{'='*60}")
    print(f"  Puzzle: {name}")
    print(f"{'='*60}")

    t0 = time.time()
    assignment, confidence, energy_hist = solve_sudoku(clues, cfg, verbose=True)
    elapsed = time.time() - t0

    valid = validate_sudoku(assignment)
    avg_conf = float(confidence.mean())
    min_conf = float(confidence.min())

    results.append({
        'name': name,
        'valid': valid,
        'time': elapsed,
        'steps': len(energy_hist),
        'avg_conf': avg_conf,
        'min_conf': min_conf,
        'final_energy': energy_hist[-1],
    })

    print(f"  Result: {'VALID' if valid else 'INVALID'}, "
          f"time={elapsed:.1f}s, steps={len(energy_hist)}, "
          f"avg_conf={avg_conf:.3f}, min_conf={min_conf:.3f}")
    print()
    print_board(assignment, clues)


In [ ]:
# --- SUMMARY TABLE (vinculum-style reporting) ---

print(f"\n{'='*70}")
print(f"  VINCULUM REPORT: solved / total = constraint satisfaction ratio")
print(f"  Each row is a vinculum: VALID_ATTEMPTS / TOTAL_ATTEMPTS")
print(f"{'='*70}")
print()

total = len(results)
solved = sum(1 for r in results if r['valid'])

print(f"{'Puzzle':<30} {'Status':<8} {'Time':<8} {'Steps':<8} {'AvgConf':<8}")
print("-" * 62)

for r in results:
    status = "OK" if r['valid'] else "--"
    print(f"{r['name']:<30} {status:<8} {r['time']:<8.1f} {r['steps']:<8} {r['avg_conf']:<8.3f}")

print()
t_str = f"{solved}/{total}"
time_str = f"{sum(r['time'] for r in results):<8.1f}"
print(f"{'TOTAL':<30} {t_str:<8} {time_str}")
print(f"  vinculum: {solved} / {total}  (solved / total puzzles)")
print(f"  mean_time: {sum(r['time'] for r in results)/total:.2f}s")
print(f"  mean_confidence: {sum(r['avg_conf'] for r in results)/total:.3f}")

In [ ]:
# --- HARDWARE REPORT ---
print(f"\n{'='*40}")
print(f"  HARDWARE REPORT")
print(f"{'='*40}")
print(f"  GPU: {GPU_INFO}")
if DEVICE.type == 'cuda':
    print(f"  CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"  CUDA memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"  Finish: {datetime.now().isoformat()}")
print(f"  Entity: Guinea Pig Trench LLC (PA, #13674084)")